# PathMNIST: FedAvg vs FedProx Under Non-IID Label Noise
### Complete GPU Execution Pipeline (Google Colab)

This notebook executes the complete experimental matrix on a GPU (T4 / A100 / V100):
1. **Pre-experimental $\mu$ sweep** on noisy seed 42 ($\mu \in \{0.001, 0.01, 0.1\}$)
2. **Automatic $\mu$ selection** using mean validation accuracy over rounds 91–100
3. **Direct reuse** of the winning sweep run as `fedprox_noisy_seed42`
4. **Full 2×2 factorial matrix** across 3 independent seeds (42, 123, 2026):
   - FedAvg Clean (Seeds 42, 123, 2026)
   - FedAvg Noisy (Seeds 42, 123, 2026)
   - FedProx Clean (Seeds 42, 123, 2026)
   - FedProx Noisy (Seeds 42, 123, 2026)
5. **Compilation of deliverables**: full numerical tables, diagnostic plots, and an executive 1-page PDF report (`PathMNIST_FedAvg_vs_FedProx_Report.pdf`).

## (Optional) Step 0: Mount Google Drive
Mounting Google Drive allows you to persist checkpoints and results directly to your Drive so nothing is lost if your Colab session disconnects.

In [ ]:
# Uncomment to mount Google Drive for persistent storage
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/PathMNIST_Results
print('Google Drive mount is optional. Running in local Colab storage by default.')

## Step 1: Verify GPU Acceleration
Ensure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU (or A100)**.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU.'
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Step 2: Install Required Dependencies
Installs MedMNIST, ReportLab for PDF generation, Matplotlib, PyTest, and Weights & Biases (`wandb`).

In [ ]:
!pip install --quiet medmnist reportlab matplotlib pytest wandb

## Step 2b: Weights & Biases (W&B) Logging Setup
Enter your W&B API key below to log live metrics, update norms, cosine similarities, and confusion matrices to project **`pathmnist-federated-learning`**.

- **Get your API key:** Visit [https://wandb.ai/authorize](https://wandb.ai/authorize)
- If you leave the key empty, the experiment will automatically run in **offline mode** and store all metrics in local JSON files.

In [ ]:
import os

# PASTE YOUR W&B API KEY HERE (or leave blank for offline logging)
WANDB_API_KEY = ""

if WANDB_API_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    import wandb
    wandb.login(key=WANDB_API_KEY.strip())
    wandb_mode = "online"
    print("W&B online logging enabled for project: pathmnist-federated-learning")
else:
    wandb_mode = "offline"
    print("No W&B key provided. Running with offline local JSON logging.")

## Step 3: Unpack Project Code
If you uploaded `colab_package.zip`, run the cell below to unzip it. (Or if running from a cloned repo, skip the unzip step).

In [ ]:
import os
if os.path.exists('colab_package.zip'):
    !unzip -q -o colab_package.zip
    print('Successfully extracted colab_package.zip')
else:
    print('Using repository directory directly.')
!ls -la

## Step 4: Generate Verified Partitions & Deterministic Initial Checkpoints
This automatically downloads official PathMNIST via `medmnist`, constructs Dirichlet ($\alpha=0.3$) splits, injects 40% noise into Client 0, prints summary tables with SHA256 hashes, and generates initial model checkpoints.

In [ ]:
!python generate_pathmnist_data.py
!python generate_initial_models.py

## Step 5: Run Unit & Correctness Test Suite
Executes 9 unit tests verifying split isolation, 40% noise on Client 0, fresh optimizer state, and the CUDA-tolerant $\mu=0$ equivalence test.

In [ ]:
!python -m pytest tests/ -v

## (Recommended) Step 5b: 5-Round Fast Pilot Run
Verifies end-to-end execution, GPU memory, all 5 clients participating, and metric logging in ~2 minutes before launching the full 15-run matrix.

In [ ]:
!python -u orchestrator.py --device cuda --pilot --wandb_mode $wandb_mode

## Step 6: Launch Full Experiment Matrix on GPU
This runs the complete workflow autonomously on CUDA:
- Phase 1: $\mu$ sweep on noisy seed 42 (0.001, 0.01, 0.1)
- Phase 2: Selects optimal $\mu$ over rounds 91–100 mean val acc
- Phase 3: Executes all 12 final runs (100 rounds each, with resume & skip-completed logic)
- Phase 4: Automatically generates tables, diagnostic plots, and the 1-page PDF report

*Expected runtime on T4 GPU: ~2 to 2.5 hours total (output streams live below).*

In [ ]:
!python -u orchestrator.py --device cuda --wandb_mode $wandb_mode

## Step 7: Inspect Final Results & Summary Tables

In [ ]:
from IPython.display import Markdown, display
with open('results/tables/results.md', 'r') as f:
    display(Markdown(f.read()))

## Step 8: View Key Diagnostic Plots Inline

In [ ]:
from IPython.display import Image, display

print('Test Accuracy Curve:')
display(Image('results/plots/plot1_test_accuracy_vs_rounds.png'))

print('Aggregation Weights:')
display(Image('results/plots/plot9_aggregation_weights.png'))

print('Mu Sweep Validation Convergence:')
display(Image('results/plots/plot_mu_sweep.png'))

## Step 9: Download All Deliverables to Local Machine
Zips and downloads the complete results directory (all JSON logs, models, tables, plots, and the executive PDF report).

In [ ]:
!zip -r PathMNIST_Final_Results.zip results/ PathMNIST_FedAvg_vs_FedProx_Report.pdf README.md DEVIATIONS.md

from google.colab import files
files.download('PathMNIST_Final_Results.zip')
files.download('PathMNIST_FedAvg_vs_FedProx_Report.pdf')